**Repository release note (2026-10-11):** ORIGINAL results/figures compilation. Table labels and some figures predate reviewer revisions; use revised manuscript and reviewer notebooks for final publication tables, confidence intervals and supplemental heatmaps.

Saved execution outputs were removed from this public copy; source calculations have not been independently rerun. Configure `OULAD_PROJECT_DIR` or use the original Colab directory.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Drive mounted and libraries loaded.")

In [ ]:
BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

RESULTS_PATH = os.path.join(
    BASE_PATH,
    "results"
)

FINAL_OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "final_outputs"
)

os.makedirs(
    FINAL_OUTPUT_PATH,
    exist_ok=True
)

print("BASE_PATH exists:",
      os.path.exists(BASE_PATH))

print("RESULTS_PATH exists:",
      os.path.exists(RESULTS_PATH))

print("FINAL_OUTPUT_PATH exists:",
      os.path.exists(FINAL_OUTPUT_PATH))

In [ ]:
result_files = sorted(
    os.listdir(RESULTS_PATH)
)

print(
    "Total result files:",
    len(result_files)
)

print("\n=== SAVED RESULT FILES ===")

for file in result_files:
    print(file)

In [ ]:
required_results = [
    # H1
    "xgboost_h1_inflation_summary.csv",
    "xgboost_all_cutoffs_performance.csv",
    "logistic_h3_all_cutoffs.csv",
    "xgboost_h3_all_cutoffs.csv",

    # H2
    "shap_family_all_cutoffs.csv",
    "shap_all_features_all_cutoffs.csv",
    "shap_key_feature_trajectory_all_cutoffs.csv",
    "final_H2_adverse_class_robustness.csv",

    # Combined summary
    "final_H1_H2_evidence_table.csv",

    # Figures already created
    "Figure_H2_Assessment_SHAP_Trajectory.png",
    "Figure_H2_Individual_Feature_SHAP_Inflation.png"
]

print("=== REQUIRED RESULT AUDIT ===")

missing_files = []

for file in required_results:

    exists = os.path.exists(
        os.path.join(
            RESULTS_PATH,
            file
        )
    )

    print(
        file,
        "->",
        exists
    )

    if not exists:
        missing_files.append(file)

print(
    "\nMissing required files:",
    missing_files
)

In [ ]:
# ==============================
# LOAD FINAL RESULT FILES
# ==============================

h1_xgb = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "xgboost_h1_inflation_summary.csv"
    )
)

xgb_performance = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "xgboost_all_cutoffs_performance.csv"
    )
)

h3_logistic = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "logistic_h3_all_cutoffs.csv"
    )
)

h3_xgb = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "xgboost_h3_all_cutoffs.csv"
    )
)

h2_family = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "shap_family_all_cutoffs.csv"
    )
)

h2_features = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "shap_all_features_all_cutoffs.csv"
    )
)

h2_adverse = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "final_H2_adverse_class_robustness.csv"
    )
)

h1_h2_evidence = pd.read_csv(
    os.path.join(
        RESULTS_PATH,
        "final_H1_H2_evidence_table.csv"
    )
)

print("All core result tables loaded successfully.")

In [ ]:
tables = {
    "H1 XGBoost": h1_xgb,
    "XGBoost performance": xgb_performance,
    "H3 Logistic": h3_logistic,
    "H3 XGBoost": h3_xgb,
    "H2 Family": h2_family,
    "H2 Features": h2_features,
    "H2 Adverse": h2_adverse,
    "H1+H2 Evidence": h1_h2_evidence
}

for name, df in tables.items():

    print("\n" + "=" * 60)
    print(name)

    print("Shape:", df.shape)

    print("Columns:")
    print(df.columns.tolist())

In [ ]:
print("=== CORE H1 + H2 RESULTS ===")

print(
    h1_h2_evidence
    .round(2)
    .to_string(index=False)
)

In [ ]:
print("=== H3 LOGISTIC COLUMNS ===")
print(h3_logistic.columns.tolist())
print("Shape:", h3_logistic.shape)

print("\n=== H3 XGBOOST COLUMNS ===")
print(h3_xgb.columns.tolist())
print("Shape:", h3_xgb.shape)

print("\n=== H3 LOGISTIC DATA ===")
print(h3_logistic.round(4).to_string(index=False))

print("\n=== H3 XGBOOST DATA ===")
print(h3_xgb.round(4).to_string(index=False))

In [ ]:
table_h1 = h1_h2_evidence[
    [
        "Cutoff",
        "LF_Accuracy",
        "Leaky_Accuracy",
        "Accuracy_Inflation_PP"
    ]
].copy()

table_h1.columns = [
    "Course Cutoff",
    "Leakage-Free Accuracy",
    "Leaky Accuracy",
    "Accuracy Inflation (pp)"
]

print("=== TABLE 1: TEMPORAL LEAKAGE AND XGBOOST PERFORMANCE ===")

print(
    table_h1
    .round(4)
    .to_string(index=False)
)

table_h1.to_csv(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Table_1_H1_XGBoost_Performance.csv"
    ),
    index=False
)

print("\nTable 1 saved.")

In [ ]:
def prepare_h3_table(df, model_name):

    temp = df[
        [
            "Cutoff",
            "Condition",
            "Fail_to_Withdrawn",
            "Withdrawn_to_Fail"
        ]
    ].copy()

    temp["Total_Fail_Withdrawn_Confusion"] = (
        temp["Fail_to_Withdrawn"]
        +
        temp["Withdrawn_to_Fail"]
    )

    temp["Model"] = model_name

    return temp


h3_logistic_clean = prepare_h3_table(
    h3_logistic,
    "Logistic Regression"
)

h3_xgb_clean = prepare_h3_table(
    h3_xgb,
    "XGBoost"
)

h3_combined = pd.concat(
    [
        h3_logistic_clean,
        h3_xgb_clean
    ],
    ignore_index=True
)

print("=== H3 CLEAN DATA ===")

print(
    h3_combined
    .to_string(index=False)
)

In [ ]:
h3_paper_rows = []

for model in [
    "Logistic Regression",
    "XGBoost"
]:

    model_data = h3_combined[
        h3_combined["Model"] == model
    ]

    for cutoff in [
        "10%",
        "20%",
        "30%",
        "40%"
    ]:

        temp = model_data[
            model_data["Cutoff"] == cutoff
        ]

        lf = temp[
            temp["Condition"] == "Leakage-Free"
        ].iloc[0]

        leaky = temp[
            temp["Condition"] == "Leaky"
        ].iloc[0]

        h3_paper_rows.append({

            "Model":
                model,

            "Cutoff":
                cutoff,

            "LF_Fail_to_Withdrawn":
                lf["Fail_to_Withdrawn"],

            "Leaky_Fail_to_Withdrawn":
                leaky["Fail_to_Withdrawn"],

            "LF_Withdrawn_to_Fail":
                lf["Withdrawn_to_Fail"],

            "Leaky_Withdrawn_to_Fail":
                leaky["Withdrawn_to_Fail"],

            "LF_Total_Confusion":
                lf["Total_Fail_Withdrawn_Confusion"],

            "Leaky_Total_Confusion":
                leaky["Total_Fail_Withdrawn_Confusion"],

            "Confusion_Change":
                (
                    leaky[
                        "Total_Fail_Withdrawn_Confusion"
                    ]
                    -
                    lf[
                        "Total_Fail_Withdrawn_Confusion"
                    ]
                )
        })


table_h3 = pd.DataFrame(
    h3_paper_rows
)

print(
    "\n=== TABLE 2: FAIL–WITHDRAWN CONFUSION ==="
)

print(
    table_h3
    .to_string(index=False)
)

table_h3.to_csv(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Table_2_H3_Fail_Withdrawn_Confusion.csv"
    ),
    index=False
)

print("\nTable 2 saved.")

In [ ]:
table_h2 = h2_family[
    [
        "Cutoff",
        "Feature_Family",
        "LF_Share_Percent",
        "Leaky_Share_Percent",
        "Share_Change_PP"
    ]
].copy()

table_h2.columns = [
    "Course Cutoff",
    "Feature Family",
    "Leakage-Free SHAP Share (%)",
    "Leaky SHAP Share (%)",
    "Change (pp)"
]

# Set clean ordering
cutoff_order = ["10%", "20%", "30%", "40%"]

family_order = [
    "Assessment",
    "In-course VLE",
    "Pre-course VLE",
    "Static/Background"
]

table_h2["Course Cutoff"] = pd.Categorical(
    table_h2["Course Cutoff"],
    categories=cutoff_order,
    ordered=True
)

table_h2["Feature Family"] = pd.Categorical(
    table_h2["Feature Family"],
    categories=family_order,
    ordered=True
)

table_h2 = (
    table_h2
    .sort_values(
        ["Course Cutoff", "Feature Family"]
    )
    .reset_index(drop=True)
)

print(
    "=== TABLE 3: SHAP EXPLANATION SHIFT BY FEATURE FAMILY ==="
)

print(
    table_h2
    .round(2)
    .to_string(index=False)
)

table_h2.to_csv(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Table_3_H2_SHAP_Family_Shift.csv"
    ),
    index=False
)

print("\nTable 3 saved.")

In [ ]:
hypothesis_summary = pd.DataFrame({

    "Hypothesis": [
        "H1",
        "H2",
        "H3"
    ],

    "Prediction": [
        "Temporal leakage inflates predictive performance",
        "Temporal leakage shifts SHAP explanations toward future assessment information",
        "Temporal leakage reduces Fail–Withdrawn confusion"
    ],

    "Main Evidence": [
        "Accuracy inflation: +16.96 to +7.98 percentage points across 10–40% cutoffs",
        "Assessment SHAP-share inflation: +31.73 to +10.22 percentage points across 10–40% cutoffs",
        "Fail–Withdrawn confusion increased rather than decreased under leakage"
    ],

    "Conclusion": [
        "Supported",
        "Supported",
        "Not supported"
    ]
})

print(
    "=== HYPOTHESIS SUMMARY ==="
)

print(
    hypothesis_summary
    .to_string(index=False)
)

hypothesis_summary.to_csv(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Hypothesis_Summary.csv"
    ),
    index=False
)

print("\nHypothesis summary saved.")

In [ ]:
print("=== TABLE 3: FINAL H2 SHAP FAMILY RESULTS ===")

print(
    table_h2
    .round(2)
    .to_string(index=False)
)

In [ ]:
# ==========================================
# FIGURE 1 — TEMPORAL LEAKAGE AND ACCURACY
# ==========================================

plot_h1 = h1_h2_evidence.copy()

plot_h1["Cutoff_Number"] = (
    plot_h1["Cutoff"]
    .str.replace("%", "", regex=False)
    .astype(int)
)

plot_h1 = plot_h1.sort_values(
    "Cutoff_Number"
)

plt.figure(figsize=(8, 5))

# Leakage-free accuracy
plt.plot(
    plot_h1["Cutoff_Number"],
    plot_h1["LF_Accuracy"],
    marker="o",
    linewidth=2,
    label="Leakage-Free"
)

# Leaky accuracy
plt.plot(
    plot_h1["Cutoff_Number"],
    plot_h1["Leaky_Accuracy"],
    marker="o",
    linewidth=2,
    label="Leaky"
)

# Annotate accuracy inflation
for _, row in plot_h1.iterrows():

    midpoint = (
        row["LF_Accuracy"]
        +
        row["Leaky_Accuracy"]
    ) / 2

    plt.text(
        row["Cutoff_Number"] + 0.5,
        midpoint,
        f'+{row["Accuracy_Inflation_PP"]:.2f} pp',
        fontsize=9
    )

plt.xlabel(
    "Course Progress Cutoff (%)"
)

plt.ylabel(
    "Classification Accuracy"
)

plt.xticks(
    [10, 20, 30, 40]
)

plt.ylim(
    0.50,
    0.76
)

plt.legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Figure_1_H1_Temporal_Leakage_Accuracy.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure 1 saved successfully."
)

In [ ]:
# ==========================================
# FIGURE 2 — FAIL/WITHDRAWN CONFUSION
# ==========================================

plot_h3 = table_h3.copy()

plot_h3["Cutoff_Number"] = (
    plot_h3["Cutoff"]
    .str.replace("%", "", regex=False)
    .astype(int)
)

plt.figure(figsize=(9, 5.5))

for model in [
    "Logistic Regression",
    "XGBoost"
]:

    temp = (
        plot_h3[
            plot_h3["Model"] == model
        ]
        .sort_values("Cutoff_Number")
    )

    plt.plot(
        temp["Cutoff_Number"],
        temp["LF_Total_Confusion"],
        marker="o",
        linewidth=2,
        linestyle="-",
        label=f"{model} — Leakage-Free"
    )

    plt.plot(
        temp["Cutoff_Number"],
        temp["Leaky_Total_Confusion"],
        marker="o",
        linewidth=2,
        linestyle="--",
        label=f"{model} — Leaky"
    )

plt.xlabel(
    "Course Progress Cutoff (%)"
)

plt.ylabel(
    "Total Fail ↔ Withdrawn Misclassifications"
)

plt.xticks(
    [10, 20, 30, 40]
)

plt.legend(
    frameon=False,
    fontsize=9
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FINAL_OUTPUT_PATH,
        "Figure_2_H3_Fail_Withdrawn_Confusion.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figure 2 saved successfully."
)

In [ ]:
import shutil

h2_figures = [
    "Figure_H2_Assessment_SHAP_Trajectory.png",
    "Figure_H2_Individual_Feature_SHAP_Inflation.png"
]

for filename in h2_figures:

    source = os.path.join(
        RESULTS_PATH,
        filename
    )

    destination = os.path.join(
        FINAL_OUTPUT_PATH,
        filename
    )

    shutil.copy2(
        source,
        destination
    )

    print(
        filename,
        "copied successfully."
    )

In [ ]:
final_required = [
    # Tables
    "Table_1_H1_XGBoost_Performance.csv",
    "Table_2_H3_Fail_Withdrawn_Confusion.csv",
    "Table_3_H2_SHAP_Family_Shift.csv",
    "Hypothesis_Summary.csv",

    # Main figures
    "Figure_1_H1_Temporal_Leakage_Accuracy.png",
    "Figure_2_H3_Fail_Withdrawn_Confusion.png",

    # H2 figures
    "Figure_H2_Assessment_SHAP_Trajectory.png",
    "Figure_H2_Individual_Feature_SHAP_Inflation.png"
]

print("=== FINAL OUTPUT AUDIT ===")

missing_final = []

for filename in final_required:

    exists = os.path.exists(
        os.path.join(
            FINAL_OUTPUT_PATH,
            filename
        )
    )

    print(
        filename,
        "->",
        exists
    )

    if not exists:
        missing_final.append(filename)

print(
    "\nMissing final outputs:",
    missing_final
)

In [ ]:
print("=== FINAL PUBLICATION PACKAGE ===")

for filename in sorted(
    os.listdir(FINAL_OUTPUT_PATH)
):
    print(filename)